In [ ]:
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd

import ev_sim
from ev_sim import plot
from ev_sim.constants import *
from ev_sim.constants import VEHICLE_STATES, RUN_DB


@dataclass
class Style:
    label: str
    color: str


styling = {
    "charge": Style("Charging", "#f5cd1d"),
    "idle": Style("Idle", "#429cf5"),
    "fm": Style("First Mile", "#f2506e"),
    "wait": Style("Waiting", "#d92344"),
    "pickup": Style("Pickup", "#d92344"),
    "lm": Style("Last Mile", "#054f75"),
    "drop": Style("Dropping", "#003049"),

    "service_level": Style("Service Level", "#003049"),
    "response_time": Style("Response Time", "#429cf5"),
    "fm_time": Style("First Mile Time", "#f2506e"),
    "lm_time": Style("Last Mile Time", "#054f75"),
}

bboxes = {
    "ann-arbor": (-83.81, 42.22, -83.67, 42.33),
    "washtenaw": (-84.17, 42.05, -83.50, 42.45),
    "se-michigan": (-84.2, 41.68, -82.4, 43.2),
    "detroit": (-83.3, 42.25, -82.9, 42.455)
}

basemaps = {
    "washtenaw": plot.BasemapStyle(base_zoom=12, labels_zoom=11),
    "se-michigan": plot.BasemapStyle(base_zoom=10, labels_zoom=9),
    "detroit": plot.BasemapStyle(base_zoom=13, labels_zoom=12)
}

line_cols = ("pick_lon", "pick_lat", "drop_lon", "drop_lat")

In [ ]:
REGION = "detroit"

bbox = bboxes.get(REGION)
basemap = basemaps.get(REGION)

df_internal = pd.read_parquet(f"output/semcog/internal-{REGION}-trips.parquet")
df_inbound = pd.read_parquet(f"output/semcog/inbound-{REGION}-trips.parquet")
df_outbound = pd.read_parquet(f"output/semcog/outbound-{REGION}-trips.parquet")

for df in (df_internal, df_inbound, df_outbound):
    for col in line_cols:
        df[col] = np.rad2deg(df[col])
        df.drop(df[~(df["created_at"] % (24 * 3600)).between(MORNING_START, MORNING_END)].index, inplace=True)

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap, blend="lighten", labels=True)
# ax.add_lines(df_internal.sample(frac=0.2), *line_cols)
ax.add_lines(df_outbound.sample(frac=0.2), *line_cols, cmap=plot.cmap_yellow, alpha=0.5)
ax.add_lines(df_inbound.sample(frac=0.2), *line_cols, cmap=plot.cmap_blue, alpha=0.5)
ax.render()

In [ ]:
REGION = "washtenaw"

bbox = bboxes.get(REGION)
basemap = basemaps.get(REGION)

df_internal = pd.read_parquet(f"output/semcog/internal-{REGION}-trips.parquet")
df_external = pd.read_parquet(f"output/semcog/external-{REGION}-trips.parquet")
df_inbound = pd.read_parquet(f"output/semcog/inbound-{REGION}-trips.parquet")
df_outbound = pd.read_parquet(f"output/semcog/outbound-{REGION}-trips.parquet")

for df in (df_internal, df_external, df_inbound, df_outbound):
    for col in line_cols:
        df[col] = np.rad2deg(df[col])
        df["time_of_day"] = pd.to_datetime(df["created_at"] % (24 * 3600), unit="s")

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap, blend="lighten", labels=True)
ax.add_lines(df_internal.sample(frac=0.2), *line_cols)
ax.add_lines(df_external.sample(frac=0.2), *line_cols, cmap=plot.cmap_blue)
ax.render()

In [ ]:
cmap_red = (
    "#1a0b0b",
    "#3a0d0d",
    "#7a1a1a",
    "#d12f2f",
    "#ff6b6b",
    "#ffc1c1",
    "#fff0f0",
)

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap, blend="lighten", labels=True)
ax.add_lines(df_outbound.sample(frac=0.2), *line_cols, cmap=cmap_red, alpha=0.5)
ax.add_lines(df_inbound.sample(frac=0.2), *line_cols, cmap=plot.cmap_blue, alpha=0.5)
ax.render()

In [ ]:
ax = plot.GeoPlot(width=1024, bbox_latlon=bbox, basemap_style=basemap, blend="lighten", labels=True)
anim = ax.animate(frame_freq="1min", fps=20, mode="tail")

anim.add_lines(df_internal, *line_cols, time="time_of_day", tail="15min")
anim.add_lines(df_external, *line_cols, cmap=plot.cmap_blue, time="time_of_day", tail="15min")

anim.save(f"output/{REGION}-requests.mp4", quality=10)

In [ ]:
ax = plot.GeoPlot(width=1024, bbox_latlon=bbox, basemap_style=basemap, blend="lighten", labels=True)
anim = ax.animate(frame_freq="1min", fps=20, mode="tail")

anim.add_lines(df_outbound, *line_cols, cmap=cmap_red, time="time_of_day", tail="15min")
anim.add_lines(df_inbound, *line_cols, cmap=plot.cmap_blue, time="time_of_day", tail="15min")

anim.save(f"output/{REGION}-external-requests.mp4", quality=10)

In [ ]:
REGION = "se-michigan"

bbox = bboxes.get(REGION)
basemap = basemaps.get(REGION)

df_all = pd.read_parquet(f"data/requests/{REGION}.parquet").sample(frac=0.2)
for col in line_cols:
    df_all[col] = np.rad2deg(df_all[col])

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap, blend="difference", labels=True)
ax.add_lines(df_all, *line_cols, cmap=plot.cmap_blue)
ax.render()

In [ ]:
REGION = "ann-arbor"

bbox = bboxes.get(REGION)
basemap = basemaps.get(REGION)

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap, blend="difference", labels=False)
ax.add_polygon(REGION_POLYGONS[REGION])
ax.render()

In [ ]:
REGION = "ann-arbor"
REGION_NAME = REGION.replace("-", " ").title()
requests_path = f"data/requests/{REGION}.parquet"

raw_requests = pd.read_parquet(requests_path)
print(len(raw_requests))
raw_requests.head()

ax = plot.Plot("# Requests versus Time", subtitle=f"     in Southeast Michigan", figsize=(10, 5))
ax.add_hist(raw_requests["created_at"], bins=7 * 24, color="#003049", alpha=1)
ax.render()

In [ ]:
db = ev_sim.RunDB(RUN_DB)

df = (db.query()
      .experiment("ann-arbor-vehicle-batching")
      .where_config("sim.length_s", "=", 24 * 3600 * 7)
      .load())
df

In [ ]:
REGION = "ann-arbor"
run = f"{REGION}-fleet-sizing/run_0012"

# ----

assert requests_path == f"data/requests/{REGION}.parquet"
run_path = f"runs/{run}"

subtitle = f"{REGION.replace("-", " ").title()}"

bbox = bboxes.get(REGION)
basemap = basemaps.get(REGION)

requests = pd.read_parquet(f"{run_path}/raw/requests.parquet")
requests = requests.merge(raw_requests, left_index=True, right_index=True, validate="1:1")

# assert (requests["start_at"] >= requests["created_at"]).all()

requests["day_of_week"], requests["time_of_day"] = requests["created_at"].divmod(24 * 3600)

completed = requests[requests["completed"]]

fleet = pd.read_parquet(f"runs/{run}/raw/fleet.parquet")
fleet.timestamp /= 3600  # to hours

waypoints = pd.read_parquet(f"runs/{run}/raw/waypoints.parquet")
waypoints["lat"] = np.rad2deg(waypoints["lat"])
waypoints["lon"] = np.rad2deg(waypoints["lon"])

# time spent per vehicle in each state (idle, fm, lm, pickup, drop, charge, service, etc.)
# cols are (idle, fm, ...) and rows are vehicles
vehicle_state_durations = ev_sim.stats.compute_vehicle_state_durations(waypoints)
vehicle_state_durations /= 3600  # to hours

completed.head()

In [ ]:
state = "idle"

cmap = {"charge": plot.cmap_yellow, "idle": plot.cmap_blue, "pickup": plot.cmap_teal}[state]

df = waypoints[waypoints["state"] == VEHICLE_STATES[state]]

ax = plot.GeoPlot(width=1000, bbox_latlon=bbox, basemap_style=basemap)
ax.add_grid(df, "lon", y="lat", bins=(200, 200), cmap=cmap)
ax.render()

In [ ]:
ax = plot.GeoPlot(width=1024, bbox_latlon=bbox, basemap_style=basemap)
anim = ax.animate(frame_freq="5min", fps=20, mode="tail")

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

anim.add_grid(df, "lon", y="lat", bins=(200, 200), cmap=cmap, time="timestamp", tail="60min")
anim.save(f"{run_path}/derived/{state}-waypoints.mp4", quality=10)

In [ ]:
ax = plot.Plot("Distribution of Vehicle Activity", subtitle=subtitle, figsize=(10, 5),
               legend=plot.LegendConfig(style="fancy"))

bins = 200
for state in ("charge", "idle", "fm", "pickup", "lm", "drop"):
    style = styling[state]
    ax.add_hist(vehicle_state_durations[state], bins=bins, label=style.label, color=style.color)
ax.show()

In [ ]:
ax = plot.Plot("Fleet Activity vs Time", subtitle="% of vehicles performing an activity over time", figsize=(10, 5),
               legend=plot.LegendConfig(style="fancy"))

cols = ["charge", "idle", "fm", "wait", "lm", "drop"]
ax.add_stackplot(fleet.timestamp, 100 * fleet[cols].T / fleet.sum(axis=1), labels=[styling[c].label for c in cols],
                 colors=[styling[c].color for c in cols])

ax.set_x_axis_unit(" hrs")
ax.set_y_axis_unit("%")

ax.show()

In [ ]:
day = "weekdays"

if day == "weekdays":
    df = requests[requests["day_of_week"] < SATURDAY]
elif day == "saturday":
    df = requests[requests["day_of_week"] == SATURDAY]
elif day == "sunday":
    df = requests[requests["day_of_week"] == SUNDAY]

hrs = tuple(range(24))
stats_vs_hr = defaultdict(list)

for hr in hrs:
    df_c = df[df["time_of_day"].between(hr * 3600, (hr + 1) * 3600)]
    hr_stats = ev_sim.compute_request_statistics(df_c)
    for k, v in hr_stats.items():
        stats_vs_hr[k].append(v)

stats_vs_hr = pd.DataFrame(stats_vs_hr)
stats_vs_hr.head()

In [ ]:
ax = plot.Plot("Metrics vs Hour of Day", subtitle=f"{subtitle}, {day.title()}", figsize=(10, 5),
               legend=plot.LegendConfig(style="fancy"))

for c in ("response_time", "fm_time"):
    style = styling[c]
    ax.add_barplot(hrs, stats_vs_hr[f"mean_{c}"], label=style.label, color=style.color)

ax.show()

In [ ]:
db = ev_sim.RunDB(RUN_DB)

df = (db.query()
      .experiment("ann-arbor-vehicle-batching")
      .where_config("sim.length_s", "=", 24 * 3600 * 7)
      .load())
df

In [ ]:
ax = plot.Plot("Response Time (mins), Service Level (%) vs Fleet Size",
               subtitle="service level is the % of customers matched with a vehicle", figsize=(10, 5),
               legend=plot.LegendConfig(style="fancy"))

style = styling["response_time"]
ax.add_line(x=df["fleet.fleet_size"].astype(int), y=df["median_response_time"], lower=df["pct90_response_time"], upper=df["pct95_response_time"],
            label=style.label, color=style.color, fill_between=True)

style = styling["service_level"]
ax.add_line(x=df["fleet.fleet_size"].astype(int), y=df["service_level"], axis=1, label=style.label, color=style.color)

ax.set_y_axis_unit(" min")
ax.set_y_axis_unit("%", axis=1)
# ax.set_x_axis_unit(" \ncars")

ax.show()

In [ ]:
ax = plot.Plot("Response Time (mins), Service Level (%) vs Fleet Size",
               subtitle="service level is the % of customers matched with a vehicle", figsize=(10, 5),
               legend=plot.LegendConfig(style="fancy"))

style = styling["response_time"]
ax.add_line(x=df["fleet.fleet_size"].astype(int), y=df["median_response_time"], lower=df["pct90_response_time"], upper=df["pct95_response_time"],
            label=style.label, color=style.color, fill_between=True)

style = styling["service_level"]
ax.add_line(x=df["fleet.fleet_size"].astype(int), y=df["service_level"], axis=1, label=style.label, color=style.color)

ax.set_y_axis_unit(" min")
ax.set_y_axis_unit("%", axis=1)
# ax.set_x_axis_unit(" \ncars")

ax.show()